# 07. Qwen2.5-7B-Instruct QLoRA Fine-Tuning Engine

**Requires GPU.** This notebook downloads and fine-tunes **Qwen/Qwen2.5-7B-Instruct**, which is fully open on HuggingFace Hub (Apache 2.0 license, no authentication needed).

Run on Colab with a GPU runtime -- see the setup cell below, which auto-clones the repo when a Colab GPU is detected.

In [ ]:
# ============================================================
# PATH & REPO AUTO-SYNC BOOSTER — Guarantees latest project code
# ============================================================
import os, sys, site, urllib.request, zipfile

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home        = os.path.expanduser('~')
proj_dir    = os.path.join(home, 'Ekegusii-LLM-Translation-main')
mistral_cfg = os.path.join(proj_dir, 'configs', 'models', 'mistral_7b.yaml')

# Auto-sync if folder is missing OR outdated (lacks mistral_7b.yaml from commit a44bf18)
if not os.path.isfile(mistral_cfg):
    print('🔄 Outdated or missing repository detected. Auto-syncing latest code from GitHub...')
    zip_path = os.path.join(home, 'repo.zip')
    urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(home)
    os.remove(zip_path)
    print('✅ Repository auto-synced to latest main commit!')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [ ]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


In [ ]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Train a single experiment. Change EXPERIMENT_ID to run a different one --
# see src.cli.train.TRAINABLE_EXPERIMENTS for the full list (E1-E7).
EXPERIMENT_ID = 'E4_Trilingual'

from src.cli.train import run_train
trainer = run_train(EXPERIMENT_ID, model_name='qwen')
print(f'Training complete. Checkpoints in checkpoints/qwen/{EXPERIMENT_ID}/')

## Run all seven experiments
Equivalent to `bash scripts/train_qwen.sh` -- trains E1 through E7 in sequence.

In [ ]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from src.cli.train import TRAINABLE_EXPERIMENTS, run_train

for experiment_id in TRAINABLE_EXPERIMENTS:
    print(f'=== Training qwen on {experiment_id} ===')
    run_train(experiment_id, model_name='qwen')